## **Import Library**

In [11]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

data = pd.read_csv("menfess_label_aug_sept.csv")

data.columns = data.columns.str.strip()

print("Ukuran data:", data.shape)
display(data.head())
display(data.dtypes)

Ukuran data: (3539, 7)


,Message_ID,Sender_ID,Date_WIB,Text,Is_relevant,topic_category,sentiment_label
0,9530,-1001647181983,2026-08-31 23:37:04,Info kost kost an cowo under 750 yg jarak nya ...,No,NaN,NaN
1,9529,-1001647181983,2026-08-31 22:57:19,Sender punya bisnis yang Alhamdulillah lolos p...,Yes,Organisasi & Pengembangan Mahasiswa,Netral
2,9528,-1001647181983,2026-08-31 22:54:49,"Ara CE maba, sumpah arek e lucu maneh, gak coc...",No,NaN,NaN
3,9527,-1001647181983,2026-08-31 22:53:57,Ada yang jual sepatu futsal gak?,No,NaN,NaN
4,9526,-1001647181983,2026-08-31 22:01:30,"infoin dong oprec DPM kapan ya, sistem pemilih...",Yes,Organisasi & Pengembangan Mahasiswa,Netral


Message_ID          int64
Sender_ID           int64
Date_WIB           object
Text               object
Is_relevant        object
topic_category     object
sentiment_label    object
dtype: object

## **Periksa Missing Value dan Duplikat**

In [12]:
print("Missing value:")
display(data.isna().sum())

print("Baris duplikat identik:", data.duplicated().sum())

print("Teks kosong:")
print(data["Text"].isna().sum())

print("Label relevansi:")
display(data["Is_relevant"].value_counts(dropna=False))

print("Kategori topik:")
display(data["topic_category"].value_counts(dropna=False))

print("Sentimen:")
display(data["sentiment_label"].value_counts(dropna=False))

Missing value:


Message_ID            0
Sender_ID             0
Date_WIB              0
Text                  0
Is_relevant           0
topic_category     2310
sentiment_label    2312
dtype: int64

Baris duplikat identik: 0
Teks kosong:
0
Label relevansi:


Is_relevant
No     2311
Yes    1228
Name: count, dtype: int64

Kategori topik:


topic_category
NaN                                      2310
Layanan & Fasilitas kampus                619
Akademik & Perkuliahan                    193
Kegiatan & Acara Kampus                   151
Organisasi & Pengembangan Mahasiswa       142
Kehidupan dan Kesejahteraan mahasiswa     124
Name: count, dtype: int64

Sentimen:


sentiment_label
NaN        2312
Netral      688
Negatif     479
Positif      60
Name: count, dtype: int64

## **Duplikat berdasarkan teks dan tanggal**

In [13]:
data["Date_WIB"] = pd.to_datetime(
    data["Date_WIB"],
    errors="coerce"
)

data["Text"] = data["Text"].astype("string")

print(
    "Duplikat berdasarkan tanggal dan teks:",
    data.duplicated(
        subset=["Date_WIB", "Text"],
        keep=False
    ).sum()
)

print(
    "Duplikat berdasarkan teks saja:",
    data.duplicated(
        subset=["Text"],
        keep=False
    ).sum()
)

duplicates = data[
    data.duplicated(
        subset=["Date_WIB", "Text"],
        keep=False
    )
].sort_values(["Date_WIB", "Text"])

display(duplicates.head(20))

Duplikat berdasarkan tanggal dan teks: 0
Duplikat berdasarkan teks saja: 4


,Message_ID,Sender_ID,Date_WIB,Text,Is_relevant,topic_category,sentiment_label


## **Menyamakan format label**

In [14]:
for col in [
    "Is_relevant",
    "topic_category",
    "sentiment_label"
]:
    data[col] = data[col].astype("string").str.strip()

data["Is_relevant"] = data["Is_relevant"].str.title()

data["sentiment_label"] = (
    data["sentiment_label"]
    .str.lower()
    .replace({
        "positif": "Positif",
        "positive": "Positif",
        "negatif": "Negatif",
        "negative": "Negatif",
        "netral": "Netral",
        "neutral": "Netral"
    })
)

display(data["Is_relevant"].value_counts(dropna=False))
display(data["sentiment_label"].value_counts(dropna=False))
display(data["topic_category"].value_counts(dropna=False))

Is_relevant
No     2311
Yes    1228
Name: count, dtype: Int64

sentiment_label
<NA>       2312
Netral      688
Negatif     479
Positif      60
Name: count, dtype: Int64

topic_category
<NA>                                     2310
Layanan & Fasilitas kampus                619
Akademik & Perkuliahan                    193
Kegiatan & Acara Kampus                   151
Organisasi & Pengembangan Mahasiswa       142
Kehidupan dan Kesejahteraan mahasiswa     124
Name: count, dtype: Int64

In [15]:
data["Date_WIB"] = pd.to_datetime(
    data["Date_WIB"], errors="coerce"
)

data["month"] = data["Date_WIB"].dt.month.map({
    8: "Agustus",
    9: "September"
})

data["month"].value_counts(dropna=False)

month
September    2078
Agustus      1461
Name: count, dtype: int64

In [16]:
print("Jumlah data:", len(data))
print("\nLabel relevansi:")
print(data["Is_relevant"].value_counts(dropna=False))

print("\nLabel sentimen:")
print(data["sentiment_label"].value_counts(dropna=False))

print("\nKategori topik:")
print(data["topic_category"].value_counts(dropna=False))

print("\nPeriode:")
print(data["month"].value_counts(dropna=False))

Jumlah data: 3539

Label relevansi:
Is_relevant
No     2311
Yes    1228
Name: count, dtype: Int64

Label sentimen:
sentiment_label
<NA>       2312
Netral      688
Negatif     479
Positif      60
Name: count, dtype: Int64

Kategori topik:
topic_category
<NA>                                     2310
Layanan & Fasilitas kampus                619
Akademik & Perkuliahan                    193
Kegiatan & Acara Kampus                   151
Organisasi & Pengembangan Mahasiswa       142
Kehidupan dan Kesejahteraan mahasiswa     124
Name: count, dtype: Int64

Periode:
month
September    2078
Agustus      1461
Name: count, dtype: int64


In [17]:
from pathlib import Path

output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)

data.to_csv(
    output_path / "data_labeled_ags_sep.csv",
    index=False
)